# Notebook 6 — Model Training, Tuning & Evaluation

In this notebook, we train and evaluate machine learning models
for late delivery prediction.

The workflow is:

1. Load the prepared train, validation, and test data.
2. Establish a naive baseline.
3. Train Logistic Regression.
4. Tune Logistic Regression hyperparameters.
5. Optimize the Logistic Regression classification threshold.
6. Train XGBoost.
7. Tune XGBoost hyperparameters.
8. Optimize the XGBoost classification threshold.
9. Compare Logistic Regression and XGBoost.
10. Evaluate a simple ensemble of both models.
11. Select the best candidate model.
12. Evaluate the selected model on the test set.

The test set is kept untouched during model training,
hyperparameter tuning, threshold optimization, and model selection.

The test set is used only once for the final evaluation.


In [12]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

from scipy.sparse import csr_matrix
from pathlib import Path

from scipy import sparse

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

In [13]:

# Features path
FEATURES_PATH = Path(r"C:\Users\user\Desktop\MLOps-Olist\data\features")


# Function to load CSR sparse matrix from NPZ
def load_sparse_matrix(path):
    data = np.load(path, allow_pickle=True)

    matrix = csr_matrix(
        (
            data["data"],
            data["indices"],
            data["indptr"]
        ),
        shape=tuple(data["shape"])
    )

    data.close()
    return matrix


# Load feature matrices
X_train = load_sparse_matrix(FEATURES_PATH / "X_train.npz")
X_validation = load_sparse_matrix(FEATURES_PATH / "X_validation.npz")
X_test = load_sparse_matrix(FEATURES_PATH / "X_test.npz")


# Load target labels
y_train = pd.read_parquet(FEATURES_PATH / "y_train.parquet")
y_validation = pd.read_parquet(FEATURES_PATH / "y_validation.parquet")
y_test = pd.read_parquet(FEATURES_PATH / "y_test.parquet")


# Convert target to 1D arrays
y_train = y_train.squeeze().to_numpy()
y_validation = y_validation.squeeze().to_numpy()
y_test = y_test.squeeze().to_numpy()


# Display shapes
print("Dataset shapes:")
print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_validation.shape, y_validation.shape)
print("Test:", X_test.shape, y_test.shape)

Dataset shapes:
Train: (67533, 17540) (67533,)
Validation: (14471, 17540) (14471,)
Test: (14472, 17540) (14472,)


## 3. Evaluation Function

Define a common evaluation function so that all models
are evaluated using the same metrics.

Because the dataset is imbalanced, we focus on:

- Precision
- Recall
- F1-score
- ROC-AUC
- Average Precision
- Accuracy

In [14]:
def evaluate_model(y_true, y_pred, y_score):

    return {
        "accuracy": accuracy_score(
            y_true,
            y_pred
        ),

        "precision": precision_score(
            y_true,
            y_pred,
            zero_division=0
        ),

        "recall": recall_score(
            y_true,
            y_pred,
            zero_division=0
        ),

        "f1": f1_score(
            y_true,
            y_pred,
            zero_division=0
        ),

        "roc_auc": roc_auc_score(
            y_true,
            y_score
        ),

        "average_precision": average_precision_score(
            y_true,
            y_score
        )
    }

## 4. Naive Baseline

Establish a simple baseline using the majority class.

This provides a reference point for judging whether
the machine learning models provide meaningful improvement.

In [15]:
baseline = DummyClassifier(
    strategy="most_frequent"
)

baseline.fit(
    X_train,
    y_train
)

baseline_pred = baseline.predict(
    X_validation
)

baseline_score = baseline.predict_proba(
    X_validation
)[:, 1]

baseline_results = evaluate_model(
    y_validation,
    baseline_pred,
    baseline_score
)

baseline_results

{'accuracy': 0.9465828208140419,
 'precision': 0.0,
 'recall': 0.0,
 'f1': 0.0,
 'roc_auc': 0.5,
 'average_precision': 0.053417179185958126}

## 5. Logistic Regression

Train a Logistic Regression model as a strong and interpretable
linear baseline.

Class weighting is used to reduce the effect of class imbalance.

In [16]:
lr_model = LogisticRegression(
    C=1.0,
    penalty="l2",
    solver="liblinear",
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)

lr_model.fit(
    X_train,
    y_train
)

lr_pred = lr_model.predict(
    X_validation
)

lr_score = lr_model.predict_proba(
    X_validation
)[:, 1]

lr_results = evaluate_model(
    y_validation,
    lr_pred,
    lr_score
)

print("Logistic Regression Results:")
display(
    pd.DataFrame([lr_results])
)

Logistic Regression Results:


,accuracy,precision,recall,f1,roc_auc,average_precision
0,0.549858,0.08052,0.712807,0.144695,0.689125,0.109793


## 6. Tune Logistic Regression

Test different values of C to find a suitable level
of regularization.

Smaller values of C apply stronger regularization,
while larger values allow the model to fit the training data more closely.

The validation set is used for model selection.

In [17]:
C_values = [
    0.001,
    0.003,
    0.01,
    0.03,
    0.1,
    0.3,
    1,
    3,
    10,
    30,
    100
]

tuning_results = []

for C in C_values:

    model = LogisticRegression(
        C=C,
        penalty="l2",
        solver="liblinear",
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    )

    model.fit(
        X_train,
        y_train
    )

    val_pred_temp = model.predict(
        X_validation
    )

    val_score_temp = model.predict_proba(
        X_validation
    )[:, 1]

    results = evaluate_model(
        y_validation,
        val_pred_temp,
        val_score_temp
    )

    results["C"] = C

    tuning_results.append(results)


tuning_df = pd.DataFrame(
    tuning_results
)

tuning_df = tuning_df[
    [
        "C",
        "accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc",
        "average_precision"
    ]
]

tuning_df = tuning_df.sort_values(
    by="f1",
    ascending=False
).reset_index(drop=True)

display(tuning_df)

,C,accuracy,precision,recall,f1,roc_auc,average_precision
0,1.000,0.549858,0.080520,0.712807,0.144695,0.689125,0.109793
1,3.000,0.594914,0.080046,0.627426,0.141979,0.669570,0.100428
2,0.300,0.454081,0.076530,0.833118,0.140183,0.716874,0.124184
3,10.000,0.611015,0.077151,0.573092,0.135994,0.657574,0.095064
4,0.100,0.368876,0.072772,0.921087,0.134887,0.734011,0.132492
5,30.000,0.615438,0.075478,0.551100,0.132772,0.651678,0.093104
6,100.000,0.617304,0.074933,0.543338,0.131703,0.647967,0.091980
7,0.030,0.302329,0.068340,0.954722,0.127549,0.738945,0.133711
8,0.010,0.272752,0.066738,0.971539,0.124896,0.738477,0.132911
9,0.003,0.248981,0.064908,0.974127,0.121707,0.738116,0.132327


## 7. Select the Best Logistic Regression Configuration

Select the C value that achieved the highest validation F1-score.

The selected C will be used for the final Logistic Regression
model before threshold optimization.

In [18]:
best_C = tuning_df.iloc[0]["C"]

print("Best C:", best_C)

Best C: 1.0


## 8. Train the Best Logistic Regression Model

Retrain Logistic Regression using the selected C value.

The validation probabilities generated by this exact model
will be used for threshold optimization.

In [19]:
best_lr = LogisticRegression(
    C=best_C,
    penalty="l2",
    solver="liblinear",
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)

best_lr.fit(
    X_train,
    y_train
)

val_score_lr = best_lr.predict_proba(
    X_validation
)[:, 1]

## 9. Optimize the Classification Threshold

The default classification threshold is 0.50.

Because the target variable is imbalanced, different thresholds
are evaluated on the validation set.

The threshold that gives the highest F1-score is selected.

In [20]:
thresholds = np.arange(
    0.10,
    0.91,
    0.01
)

threshold_results = []

for threshold in thresholds:

    pred = (
        val_score_lr >= threshold
    ).astype(int)

    threshold_results.append({
        "threshold": round(threshold, 2),

        "accuracy": accuracy_score(
            y_validation,
            pred
        ),

        "precision": precision_score(
            y_validation,
            pred,
            zero_division=0
        ),

        "recall": recall_score(
            y_validation,
            pred,
            zero_division=0
        ),

        "f1": f1_score(
            y_validation,
            pred,
            zero_division=0
        )
    })


threshold_df = pd.DataFrame(
    threshold_results
)

threshold_df = threshold_df.sort_values(
    by="f1",
    ascending=False
).reset_index(drop=True)

best_threshold = threshold_df.iloc[0]["threshold"]

print("Best C:", best_C)
print("Best Threshold:", best_threshold)

display(
    threshold_df.head(10)
)

Best C: 1.0
Best Threshold: 0.81


,threshold,accuracy,precision,recall,f1
0,0.81,0.843480,0.131058,0.342820,0.189624
1,0.82,0.852740,0.134159,0.322122,0.189426
2,0.79,0.825098,0.125639,0.381630,0.189042
3,0.80,0.835119,0.128170,0.359638,0.188987
4,0.78,0.815148,0.122619,0.399741,0.187671
5,0.83,0.860825,0.135215,0.297542,0.185934
6,0.77,0.804091,0.118431,0.413972,0.184173
7,0.84,0.870085,0.137999,0.272962,0.183319
8,0.76,0.795315,0.116369,0.429495,0.183122
9,0.75,0.784880,0.113606,0.445019,0.181005


## 10. Evaluate Logistic Regression on Validation Data

Evaluate the selected Logistic Regression model using
the selected classification threshold.

This includes the classification report and confusion matrix.

In [21]:
val_pred_lr_final = (
    val_score_lr >= best_threshold
).astype(int)

print("Final Logistic Regression")
print("-" * 50)

print(f"C: {best_C}")
print(f"Threshold: {best_threshold}")

print("\nClassification Report:")

print(
    classification_report(
        y_validation,
        val_pred_lr_final,
        target_names=[
            "On Time",
            "Late"
        ],
        zero_division=0
    )
)

cm_lr = confusion_matrix(
    y_validation,
    val_pred_lr_final
)

print("\nConfusion Matrix:")
print(cm_lr)

Final Logistic Regression
--------------------------------------------------
C: 1.0
Threshold: 0.81

Classification Report:
              precision    recall  f1-score   support

     On Time       0.96      0.87      0.91     13698
        Late       0.13      0.34      0.19       773

    accuracy                           0.84     14471
   macro avg       0.55      0.61      0.55     14471
weighted avg       0.91      0.84      0.87     14471


Confusion Matrix:
[[11941  1757]
 [  508   265]]


## 11. Store Logistic Regression Results

Store the final Logistic Regression validation metrics
so they can later be compared with XGBoost and other models.

In [22]:
final_lr_results = evaluate_model(
    y_validation,
    val_pred_lr_final,
    val_score_lr
)

final_lr_results["model"] = "Logistic Regression"
final_lr_results["C"] = best_C
final_lr_results["threshold"] = best_threshold

display(
    pd.DataFrame([final_lr_results])
)

,accuracy,precision,recall,f1,roc_auc,average_precision,model,C,threshold
0,0.84348,0.131058,0.34282,0.189624,0.689125,0.109793,Logistic Regression,1.0,0.81


## 12. XGBoost

Train an XGBoost classifier as a nonlinear candidate model.

XGBoost can capture nonlinear relationships and interactions
between features that Logistic Regression may not capture.

In [23]:
from xgboost import XGBClassifier

## 13. Train Initial XGBoost Model

Train an initial XGBoost model using the training data.

The validation set is used to evaluate the model before tuning.

In [24]:
xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train,
    y_train
)

xgb_score = xgb_model.predict_proba(
    X_validation
)[:, 1]

xgb_pred = (
    xgb_score >= 0.5
).astype(int)

xgb_results = evaluate_model(
    y_validation,
    xgb_pred,
    xgb_score
)

print("Initial XGBoost Results:")
display(
    pd.DataFrame([xgb_results])
)

Initial XGBoost Results:


,accuracy,precision,recall,f1,roc_auc,average_precision
0,0.946445,0.428571,0.007762,0.015248,0.745655,0.165974


## 14. Optimize the XGBoost Classification Threshold

The default threshold of 0.50 produces very low recall for the
Late class.

We therefore evaluate multiple thresholds using the validation set
and select the threshold that gives the highest F1-score.

In [25]:
thresholds = np.arange(
    0.10,
    0.91,
    0.01
)

xgb_threshold_results = []

for threshold in thresholds:

    pred = (
        xgb_score >= threshold
    ).astype(int)

    xgb_threshold_results.append({
        "threshold": round(threshold, 2),

        "accuracy": accuracy_score(
            y_validation,
            pred
        ),

        "precision": precision_score(
            y_validation,
            pred,
            zero_division=0
        ),

        "recall": recall_score(
            y_validation,
            pred,
            zero_division=0
        ),

        "f1": f1_score(
            y_validation,
            pred,
            zero_division=0
        )
    })


xgb_threshold_df = pd.DataFrame(
    xgb_threshold_results
)

xgb_threshold_df = xgb_threshold_df.sort_values(
    by="f1",
    ascending=False
).reset_index(drop=True)

best_xgb_threshold = (
    xgb_threshold_df.iloc[0]["threshold"]
)

print(
    "Best XGBoost Threshold:",
    best_xgb_threshold
)

display(
    xgb_threshold_df.head(10)
)

Best XGBoost Threshold: 0.13


,threshold,accuracy,precision,recall,f1
0,0.13,0.895170,0.199515,0.319534,0.245649
1,0.14,0.901873,0.207769,0.297542,0.244681
2,0.12,0.886670,0.187004,0.335058,0.240037
3,0.15,0.907125,0.211325,0.270375,0.237230
4,0.11,0.876443,0.176957,0.359638,0.237201
5,0.16,0.911893,0.219866,0.254851,0.236070
6,0.10,0.863589,0.168415,0.394567,0.236068
7,0.17,0.914311,0.216970,0.231565,0.224030
8,0.18,0.917490,0.221192,0.216041,0.218586
9,0.20,0.923917,0.241325,0.197930,0.217484


## 15. Tune XGBoost Hyperparameters

Tune the main XGBoost hyperparameters to improve the model's
ability to distinguish between On Time and Late deliveries.

The validation set is used for model selection.

In [26]:
xgb_configs = [
    {
        "n_estimators": 300,
        "max_depth": 3,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8
    },
    {
        "n_estimators": 500,
        "max_depth": 3,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8
    },
    {
        "n_estimators": 300,
        "max_depth": 4,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8
    },
    {
        "n_estimators": 500,
        "max_depth": 4,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8
    },
    {
        "n_estimators": 300,
        "max_depth": 5,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8
    },
    {
        "n_estimators": 500,
        "max_depth": 5,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8
    }
]

xgb_tuning_results = []

for config in xgb_configs:

    model = XGBClassifier(
        n_estimators=config["n_estimators"],
        max_depth=config["max_depth"],
        learning_rate=config["learning_rate"],
        subsample=config["subsample"],
        colsample_bytree=config["colsample_bytree"],
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train,
        y_train
    )

    score = model.predict_proba(
        X_validation
    )[:, 1]

    pred = (
        score >= 0.5
    ).astype(int)

    results = evaluate_model(
        y_validation,
        pred,
        score
    )

    results.update(config)

    xgb_tuning_results.append(results)


xgb_tuning_df = pd.DataFrame(
    xgb_tuning_results
)

xgb_tuning_df = xgb_tuning_df[
    [
        "n_estimators",
        "max_depth",
        "learning_rate",
        "subsample",
        "colsample_bytree",
        "accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc",
        "average_precision"
    ]
]

xgb_tuning_df = xgb_tuning_df.sort_values(
    by="average_precision",
    ascending=False
).reset_index(drop=True)

display(
    xgb_tuning_df
)

,n_estimators,max_depth,learning_rate,subsample,colsample_bytree,accuracy,precision,recall,f1,roc_auc,average_precision
0,300,5,0.05,0.8,0.8,0.946514,0.454545,0.006468,0.012755,0.752926,0.166627
1,500,5,0.05,0.8,0.8,0.946306,0.388889,0.009056,0.017699,0.750758,0.164341
2,500,4,0.05,0.8,0.8,0.946583,0.500000,0.006468,0.012771,0.755802,0.163086
3,300,4,0.05,0.8,0.8,0.946514,0.400000,0.002587,0.005141,0.754853,0.162433
4,500,3,0.05,0.8,0.8,0.946583,0.500000,0.007762,0.015287,0.752924,0.160478
5,300,3,0.05,0.8,0.8,0.946721,0.666667,0.005175,0.010270,0.753265,0.158332


## 16. Train the Selected XGBoost Configuration

Train XGBoost using the selected hyperparameters.

The validation probabilities from this exact model will be used
for the final threshold optimization.

In [27]:
best_xgb = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

best_xgb.fit(
    X_train,
    y_train
)

val_score_xgb = best_xgb.predict_proba(
    X_validation
)[:, 1]

print("Selected XGBoost Configuration")
print("-" * 50)
print("n_estimators:", 500)
print("max_depth:", 4)
print("learning_rate:", 0.05)
print("subsample:", 0.8)
print("colsample_bytree:", 0.8)

Selected XGBoost Configuration
--------------------------------------------------
n_estimators: 500
max_depth: 4
learning_rate: 0.05
subsample: 0.8
colsample_bytree: 0.8


## 17. Optimize the XGBoost Classification Threshold

Evaluate different classification thresholds using the validation set.

The threshold with the highest F1-score will be selected.

In [28]:
thresholds = np.arange(
    0.05,
    0.51,
    0.01
)

xgb_final_threshold_results = []

for threshold in thresholds:

    pred = (
        val_score_xgb >= threshold
    ).astype(int)

    xgb_final_threshold_results.append({
        "threshold": round(threshold, 2),

        "accuracy": accuracy_score(
            y_validation,
            pred
        ),

        "precision": precision_score(
            y_validation,
            pred,
            zero_division=0
        ),

        "recall": recall_score(
            y_validation,
            pred,
            zero_division=0
        ),

        "f1": f1_score(
            y_validation,
            pred,
            zero_division=0
        )
    })


xgb_final_threshold_df = pd.DataFrame(
    xgb_final_threshold_results
)

xgb_final_threshold_df = (
    xgb_final_threshold_df
    .sort_values(
        by="f1",
        ascending=False
    )
    .reset_index(drop=True)
)

best_xgb_threshold = (
    xgb_final_threshold_df.iloc[0]["threshold"]
)

print(
    "Best XGBoost Threshold:",
    best_xgb_threshold
)

display(
    xgb_final_threshold_df.head(10)
)

Best XGBoost Threshold: 0.15


,threshold,accuracy,precision,recall,f1
0,0.15,0.899730,0.206239,0.307891,0.247016
1,0.14,0.892751,0.194988,0.322122,0.242927
2,0.16,0.906434,0.212661,0.278137,0.241031
3,0.13,0.882109,0.181135,0.342820,0.237030
4,0.12,0.870845,0.169880,0.364812,0.231813
5,0.17,0.910511,0.213187,0.250970,0.230541
6,0.09,0.819017,0.145817,0.491591,0.224919
7,0.18,0.914035,0.216606,0.232859,0.224439
8,0.11,0.857646,0.158258,0.385511,0.224398
9,0.10,0.839057,0.149550,0.429495,0.221851


## 18. Final XGBoost Validation Evaluation

Evaluate the selected XGBoost model using the optimized
classification threshold.

The evaluation includes the classification report and
confusion matrix.

In [29]:
# Final XGBoost predictions using the selected threshold

val_pred_xgb_final = (
    val_score_xgb >= best_xgb_threshold
).astype(int)

print("Final XGBoost")
print("-" * 50)

print(f"n_estimators: 500")
print(f"max_depth: 4")
print(f"learning_rate: 0.05")
print(f"subsample: 0.8")
print(f"colsample_bytree: 0.8")
print(f"Threshold: {best_xgb_threshold}")

print("\nClassification Report:")

print(
    classification_report(
        y_validation,
        val_pred_xgb_final,
        target_names=[
            "On Time",
            "Late"
        ],
        zero_division=0
    )
)

cm_xgb = confusion_matrix(
    y_validation,
    val_pred_xgb_final
)

print("\nConfusion Matrix:")
print(cm_xgb)

Final XGBoost
--------------------------------------------------
n_estimators: 500
max_depth: 4
learning_rate: 0.05
subsample: 0.8
colsample_bytree: 0.8
Threshold: 0.15

Classification Report:
              precision    recall  f1-score   support

     On Time       0.96      0.93      0.95     13698
        Late       0.21      0.31      0.25       773

    accuracy                           0.90     14471
   macro avg       0.58      0.62      0.60     14471
weighted avg       0.92      0.90      0.91     14471


Confusion Matrix:
[[12782   916]
 [  535   238]]


## 19. Compare Logistic Regression and XGBoost

Compare the final validation results of both candidate models.

The comparison focuses on Precision, Recall, F1-score,
ROC-AUC, and Average Precision.

The Test set is still not used.

In [30]:
# Store final XGBoost validation results

final_xgb_results = evaluate_model(
    y_validation,
    val_pred_xgb_final,
    val_score_xgb
)

final_xgb_results["model"] = "XGBoost"
final_xgb_results["n_estimators"] = 500
final_xgb_results["max_depth"] = 4
final_xgb_results["learning_rate"] = 0.05
final_xgb_results["threshold"] = best_xgb_threshold


# Combine both models

comparison_df = pd.DataFrame([
    final_lr_results,
    final_xgb_results
])


# Select useful columns

comparison_df = comparison_df[
    [
        "model",
        "accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc",
        "average_precision",
        "threshold"
    ]
]


display(comparison_df) 

,model,accuracy,precision,recall,f1,roc_auc,average_precision,threshold
0,Logistic Regression,0.84348,0.131058,0.342820,0.189624,0.689125,0.109793,0.81
1,XGBoost,0.89973,0.206239,0.307891,0.247016,0.755802,0.163086,0.15


## 20. Logistic Regression + XGBoost Ensemble

Combine the prediction probabilities of Logistic Regression
and XGBoost using a weighted average.

Different weights and classification thresholds are tested
using the validation set.

The Test set remains untouched.

In [31]:
# Candidate weights for Logistic Regression
# The remaining weight is assigned to XGBoost

lr_weights = np.arange(
    0.0,
    1.01,
    0.1
)

ensemble_results = []

thresholds = np.arange(
    0.05,
    0.91,
    0.01
)


for lr_weight in lr_weights:

    xgb_weight = 1 - lr_weight

    # Combine model probabilities
    ensemble_score = (
        lr_weight * val_score_lr
        + xgb_weight * val_score_xgb
    )

    # Find the best threshold for this weight
    for threshold in thresholds:

        ensemble_pred = (
            ensemble_score >= threshold
        ).astype(int)

        ensemble_results.append({
            "lr_weight": round(lr_weight, 2),
            "xgb_weight": round(xgb_weight, 2),
            "threshold": round(threshold, 2),

            "accuracy": accuracy_score(
                y_validation,
                ensemble_pred
            ),

            "precision": precision_score(
                y_validation,
                ensemble_pred,
                zero_division=0
            ),

            "recall": recall_score(
                y_validation,
                ensemble_pred,
                zero_division=0
            ),

            "f1": f1_score(
                y_validation,
                ensemble_pred,
                zero_division=0
            ),

            "roc_auc": roc_auc_score(
                y_validation,
                ensemble_score
            ),

            "average_precision": average_precision_score(
                y_validation,
                ensemble_score
            )
        })


ensemble_df = pd.DataFrame(
    ensemble_results
)


# Rank by F1

ensemble_df = ensemble_df.sort_values(
    by="f1",
    ascending=False
).reset_index(drop=True)


display(
    ensemble_df.head(10)
) 

,lr_weight,xgb_weight,threshold,accuracy,precision,recall,f1,roc_auc,average_precision
0,0.0,1.0,0.15,0.899730,0.206239,0.307891,0.247016,0.755802,0.163086
1,0.0,1.0,0.14,0.892751,0.194988,0.322122,0.242927,0.755802,0.163086
2,0.0,1.0,0.16,0.906434,0.212661,0.278137,0.241031,0.755802,0.163086
3,0.0,1.0,0.13,0.882109,0.181135,0.342820,0.237030,0.755802,0.163086
4,0.1,0.9,0.22,0.905397,0.206114,0.270375,0.233912,0.754502,0.163092
5,0.1,0.9,0.21,0.898003,0.195143,0.291074,0.233645,0.754502,0.163092
6,0.1,0.9,0.19,0.880934,0.177748,0.338939,0.233200,0.754502,0.163092
7,0.1,0.9,0.20,0.889918,0.185583,0.313066,0.233028,0.754502,0.163092
8,0.0,1.0,0.12,0.870845,0.169880,0.364812,0.231813,0.755802,0.163086
9,0.1,0.9,0.23,0.910994,0.214839,0.250970,0.231504,0.754502,0.163092


## 21. Select the Final Model

Based on the validation results, XGBoost is selected as the
final model.

The model achieved the highest F1-score among the tested
Logistic Regression, XGBoost, and ensemble configurations.

The selected classification threshold is 0.15.

In [32]:
# Select XGBoost as the final model

final_model = best_xgb

final_threshold = best_xgb_threshold

final_model_name = "XGBoost"

print("Final Model Selection")
print("-" * 50)

print("Model:", final_model_name)
print("n_estimators:", 500)
print("max_depth:", 4)
print("learning_rate:", 0.05)
print("subsample:", 0.8)
print("colsample_bytree:", 0.8)
print("Threshold:", final_threshold)

print("\nValidation Performance")
print("-" * 50)

print(f"Accuracy: {final_xgb_results['accuracy']:.4f}")
print(f"Precision: {final_xgb_results['precision']:.4f}")
print(f"Recall: {final_xgb_results['recall']:.4f}")
print(f"F1-score: {final_xgb_results['f1']:.4f}")
print(f"ROC-AUC: {final_xgb_results['roc_auc']:.4f}")
print(f"Average Precision: {final_xgb_results['average_precision']:.4f}")

Final Model Selection
--------------------------------------------------
Model: XGBoost
n_estimators: 500
max_depth: 4
learning_rate: 0.05
subsample: 0.8
colsample_bytree: 0.8
Threshold: 0.15

Validation Performance
--------------------------------------------------
Accuracy: 0.8997
Precision: 0.2062
Recall: 0.3079
F1-score: 0.2470
ROC-AUC: 0.7558
Average Precision: 0.1631


## 22. Final Model Evaluation on Test Set

Evaluate the selected XGBoost model on the untouched test set.

The optimized threshold selected from the validation set
is applied to the test predictions.

This is the final unbiased evaluation of the model.

In [33]:
# Generate probability scores on the untouched test set

test_score = final_model.predict_proba(
    X_test
)[:, 1]


# Apply the threshold selected using validation data

test_pred = (
    test_score >= final_threshold
).astype(int)


# Evaluate the final model

final_test_results = evaluate_model(
    y_test,
    test_pred,
    test_score
)


print("Final XGBoost — Test Results")
print("-" * 50)

print(f"Threshold: {final_threshold}")

print(f"Accuracy: {final_test_results['accuracy']:.4f}")
print(f"Precision: {final_test_results['precision']:.4f}")
print(f"Recall: {final_test_results['recall']:.4f}")
print(f"F1-score: {final_test_results['f1']:.4f}")
print(f"ROC-AUC: {final_test_results['roc_auc']:.4f}")
print(f"Average Precision: {final_test_results['average_precision']:.4f}")

Final XGBoost — Test Results
--------------------------------------------------
Threshold: 0.15
Accuracy: 0.8184
Precision: 0.1211
Recall: 0.2790
F1-score: 0.1689
ROC-AUC: 0.6716
Average Precision: 0.1188


## 23. Advanced XGBoost Tuning

The initial XGBoost model showed promising validation performance,
but its performance decreased on the test set.

In this stage, we perform additional hyperparameter tuning
to improve the model's ability to identify late deliveries.

Only the training and validation sets are used during this stage.
The test set remains completely untouched.


In [34]:
# Advanced XGBoost configurations

advanced_xgb_configs = [
    {
        "n_estimators": 500,
        "max_depth": 3,
        "learning_rate": 0.05,
        "min_child_weight": 1,
        "gamma": 0,
        "reg_alpha": 0,
        "reg_lambda": 1,
        "scale_pos_weight": 1
    },
    {
        "n_estimators": 500,
        "max_depth": 4,
        "learning_rate": 0.05,
        "min_child_weight": 1,
        "gamma": 0,
        "reg_alpha": 0,
        "reg_lambda": 1,
        "scale_pos_weight": 1
    },
    {
        "n_estimators": 500,
        "max_depth": 4,
        "learning_rate": 0.05,
        "min_child_weight": 3,
        "gamma": 0,
        "reg_alpha": 0,
        "reg_lambda": 1,
        "scale_pos_weight": 1
    },
    {
        "n_estimators": 500,
        "max_depth": 4,
        "learning_rate": 0.05,
        "min_child_weight": 5,
        "gamma": 0,
        "reg_alpha": 0,
        "reg_lambda": 1,
        "scale_pos_weight": 1
    },
    {
        "n_estimators": 500,
        "max_depth": 4,
        "learning_rate": 0.05,
        "min_child_weight": 3,
        "gamma": 0.1,
        "reg_alpha": 0,
        "reg_lambda": 1,
        "scale_pos_weight": 1
    },
    {
        "n_estimators": 500,
        "max_depth": 4,
        "learning_rate": 0.05,
        "min_child_weight": 3,
        "gamma": 0,
        "reg_alpha": 0.1,
        "reg_lambda": 1,
        "scale_pos_weight": 1
    },
    {
        "n_estimators": 500,
        "max_depth": 4,
        "learning_rate": 0.05,
        "min_child_weight": 3,
        "gamma": 0,
        "reg_alpha": 0,
        "reg_lambda": 2,
        "scale_pos_weight": 1
    },
    {
        "n_estimators": 500,
        "max_depth": 4,
        "learning_rate": 0.05,
        "min_child_weight": 3,
        "gamma": 0,
        "reg_alpha": 0,
        "reg_lambda": 1,
        "scale_pos_weight": 2
    }
]

advanced_results = []

for config in advanced_xgb_configs:

    model = XGBClassifier(
        n_estimators=config["n_estimators"],
        max_depth=config["max_depth"],
        learning_rate=config["learning_rate"],
        min_child_weight=config["min_child_weight"],
        gamma=config["gamma"],
        reg_alpha=config["reg_alpha"],
        reg_lambda=config["reg_lambda"],
        scale_pos_weight=config["scale_pos_weight"],
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train,
        y_train
    )

    score = model.predict_proba(
        X_validation
    )[:, 1]

    pred = (
        score >= 0.5
    ).astype(int)

    results = evaluate_model(
        y_validation,
        pred,
        score
    )

    results.update(config)

    advanced_results.append(results)


advanced_xgb_df = pd.DataFrame(
    advanced_results
)

advanced_xgb_df = advanced_xgb_df[
    [
        "n_estimators",
        "max_depth",
        "learning_rate",
        "min_child_weight",
        "gamma",
        "reg_alpha",
        "reg_lambda",
        "scale_pos_weight",
        "accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc",
        "average_precision"
    ]
]

advanced_xgb_df = advanced_xgb_df.sort_values(
    by="average_precision",
    ascending=False
).reset_index(drop=True)

display(advanced_xgb_df)

,n_estimators,max_depth,learning_rate,min_child_weight,gamma,reg_alpha,reg_lambda,scale_pos_weight,accuracy,precision,recall,f1,roc_auc,average_precision
0,500,4,0.05,5,0.0,0.0,1,1,0.946514,0.454545,0.006468,0.012755,0.756376,0.165674
1,500,4,0.05,3,0.0,0.1,1,1,0.946445,0.400000,0.005175,0.010217,0.757369,0.164566
2,500,4,0.05,3,0.0,0.0,1,1,0.946514,0.454545,0.006468,0.012755,0.755406,0.164132
3,500,4,0.05,3,0.1,0.0,1,1,0.946376,0.384615,0.006468,0.012723,0.753554,0.163394
4,500,4,0.05,1,0.0,0.0,1,1,0.946583,0.500000,0.006468,0.012771,0.755802,0.163086
5,500,4,0.05,3,0.0,0.0,1,2,0.943473,0.307692,0.046572,0.080899,0.749492,0.160901
6,500,3,0.05,1,0.0,0.0,1,1,0.946583,0.500000,0.007762,0.015287,0.752924,0.160478
7,500,4,0.05,3,0.0,0.0,2,1,0.946583,0.500000,0.005175,0.010243,0.752595,0.160113


## 24. Advanced XGBoost Threshold Tuning

The best advanced XGBoost configuration is evaluated across
different classification thresholds.

The threshold is optimized using the validation set only.

The goal is to find a better balance between Precision and Recall
and maximize the F1-score.


In [35]:
# Best advanced XGBoost configuration

advanced_best_model = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.05,
    min_child_weight=5,
    gamma=0,
    reg_alpha=0,
    reg_lambda=1,
    scale_pos_weight=1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

advanced_best_model.fit(
    X_train,
    y_train
)


# Validation probabilities

advanced_val_score = advanced_best_model.predict_proba(
    X_validation
)[:, 1]


# Threshold tuning

thresholds = np.arange(
    0.05,
    0.51,
    0.01
)

advanced_threshold_results = []

for threshold in thresholds:

    pred = (
        advanced_val_score >= threshold
    ).astype(int)

    advanced_threshold_results.append({
        "threshold": round(threshold, 2),

        "accuracy": accuracy_score(
            y_validation,
            pred
        ),

        "precision": precision_score(
            y_validation,
            pred,
            zero_division=0
        ),

        "recall": recall_score(
            y_validation,
            pred,
            zero_division=0
        ),

        "f1": f1_score(
            y_validation,
            pred,
            zero_division=0
        )
    })


advanced_threshold_df = pd.DataFrame(
    advanced_threshold_results
)

advanced_threshold_df = (
    advanced_threshold_df
    .sort_values(
        by="f1",
        ascending=False
    )
    .reset_index(drop=True)
)


best_advanced_threshold = (
    advanced_threshold_df.iloc[0]["threshold"]
)


print(
    "Best Advanced XGBoost Threshold:",
    best_advanced_threshold
)

display(
    advanced_threshold_df.head(10)
)

Best Advanced XGBoost Threshold: 0.16


,threshold,accuracy,precision,recall,f1
0,0.16,0.905812,0.211350,0.279431,0.240669
1,0.17,0.911133,0.220892,0.262613,0.239953
2,0.15,0.900007,0.201770,0.294955,0.239622
3,0.14,0.892406,0.191824,0.315653,0.238631
4,0.11,0.862069,0.168923,0.403622,0.238168
5,0.12,0.874162,0.175341,0.366106,0.237118
6,0.18,0.916177,0.229730,0.241915,0.235665
7,0.13,0.883353,0.180740,0.335058,0.234814
8,0.10,0.842927,0.152778,0.426908,0.225026
9,0.19,0.919287,0.229822,0.217335,0.223404


## 25. Notebook Summary

Several classification approaches were evaluated for late delivery
prediction.

Logistic Regression and XGBoost were trained and tuned using the
training and validation sets. A simple ensemble was also evaluated.

XGBoost achieved the best validation performance, with an F1-score
of 24.70%, ROC-AUC of 75.58%, and Average Precision of 16.31%.

Additional XGBoost tuning was performed, but it did not improve
the validation F1-score.

Therefore, the original XGBoost configuration was retained as the
best candidate model.

The test set was kept isolated from model selection and tuning.
Final model training using the combined training and validation
data will be performed in the next notebook.
